# Dependency Management with uv — Bob Discovers the Force of uv

*One reformed admin, one project, three files: `pyproject.toml`, `uv.lock`, `.venv`.*

## Introduction

> **Bob:** Alice says uv can keep my dependencies predictable. My runbook needs to show what happens when I add a package, change a constraint, or ship the result.

This notebook follows [Bob's first uv project](../uv_proj_scaffolding_lab/uv_proj_scaffolding_lab.ipynb). Run it top to bottom in its own Dev Container at `/workspace`. It accompanies [Dependency Management with uv](../../docs/chapter-03/section-02.md).

| File | Bob's mental model |
| --- | --- |
| `pyproject.toml` | **Intent** — declared packages and constraints |
| `uv.lock` | **Truth** — exact resolved versions and sources |
| `.venv` | **Reality** — what is installed |

Check that the container has uv:

In [ ]:
%%bash
set -euxo pipefail
uv --version

Before starting, clear only the artifacts generated by an earlier run of this lab:

In [ ]:
%%bash
set -euxo pipefail
rm -rf pyproject.toml uv.lock .venv installed-demo local-helper resolution-demo requirements.txt

## Declaring Dependencies

> **Bob:** No more mystery installs on a server. First I declare what my license service needs.

Start with the same metadata-only `license-service` project as the section. `uv add` and `uv remove` will update this file, resolve a lockfile, and sync `.venv`:

In [ ]:
%%bash
set -euxo pipefail
cat > pyproject.toml <<'EOF'
[project]
name = "license-service"
version = "0.1.0"
requires-python = ">=3.11"
EOF
cat pyproject.toml

### Add Dependencies

#### Add from a Package Index

> **Bob:** So `uv add` writes my intent, locks the versions, *and* installs them? Suspiciously convenient.

Add a minimum-version dependency, an exact pin, and a range allowing patch releases in the FastAPI 0.115 series. The chosen `requests` lower bound may vary by date:

In [ ]:
%%bash
set -euxo pipefail
uv add requests
uv add "httpx==0.27.2"
uv add "fastapi>=0.115,<0.116"
cat pyproject.toml

The file now lists `requests>=…`, `httpx==0.27.2`, and `fastapi>=0.115,<0.116`. Use ranges when compatible updates are welcome; the lockfile supplies exact versions for reproducible installs. Bob keeps the `httpx` pin because he will test what happens when a constraint cannot move.

#### Add a Dependency from a Git Repository

> **Bob:** Alice's example uses Flask from GitHub. I want to see where uv writes the URL — and keep an offline-friendly exercise for the runbook.

The section's command is `uv add "flask @ git+https://github.com/pallets/flask"`. It puts `flask` under `[project].dependencies` and its URL under `[tool.uv.sources]`. Try that same example here (requires GitHub access):

In [ ]:
%%bash
set -euxo pipefail
uv add "flask @ git+https://github.com/pallets/flask"
cat pyproject.toml

For a second look at Git sources, Bob makes a tiny local Git package. `--no-workspace` prevents the helper from joining the main project; its Git URL goes into the same source table.

In [ ]:
%%bash
set -euxo pipefail
uv init --lib --no-workspace --vcs none --author-from none --python 3.11 local-helper
git -C local-helper -c user.name=Bob -c user.email=bob@example.invalid init -q
git -C local-helper add .
git -C local-helper -c user.name=Bob -c user.email=bob@example.invalid commit -qm 'Add local helper'
uv add "local-helper @ git+file://$(pwd)/local-helper"
cat pyproject.toml

Remove only the extra helper so the rest of the lab follows the section's Flask example:

In [ ]:
%%bash
set -euxo pipefail
uv remove local-helper
cat pyproject.toml

#### Dependency Groups

> **Bob:** Test tools and docs generators don't belong in the runtime list. Give them their own drawers.

Add `pytest` and `ruff` to `dev` (`--dev` is shorthand for `--group dev`) and MkDocs to `docs`. The `dev` group is installed by default; `docs` needs explicit selection:

In [ ]:
%%bash
set -euxo pipefail
uv add --group dev pytest
uv add --dev ruff
uv add --group docs "mkdocs>=1.6.1,<2.0"
cat pyproject.toml

#### Dependency Markers

> **Bob:** Windows Integrated Authentication needs `pywin32`, but this Linux container doesn't. How do I declare the Windows requirement without installing it here?

Use a PEP 508 marker; uv records `pywin32` for Windows while skipping its installation on Linux:

In [ ]:
%%bash
set -euxo pipefail
uv add "pywin32>=310" --marker "sys_platform == 'win32'"
cat pyproject.toml

### Remove Dependencies

> **Bob:** If I remove a package, uv should clean up the declaration, lockfile, and environment together.

Remove the runtime `requests` dependency and `pytest` from `dev`, as in the section. Flask, FastAPI, HTTPX, the marker, `ruff`, and MkDocs remain:

In [ ]:
%%bash
set -euxo pipefail
uv remove requests
uv remove --group dev pytest
cat pyproject.toml

## Locking

### The Lockfile

> **Bob:** `pyproject.toml` says *which versions are allowed*. `uv.lock` tells me exactly which ones were resolved, including transitive packages like Starlette and AnyIO.

`uv lock` resolves without installing or creating an environment; `uv add` has already created one here. Re-running `uv lock` should preserve compatible locked versions. The file also records sources and download hashes for supported environments:

In [ ]:
%%bash
set -euxo pipefail
uv lock
ls -la uv.lock
python3 - <<'PY'
import tomllib
with open('uv.lock', 'rb') as file:
    lock = tomllib.load(file)
print('Lock format:', lock['version'], 'revision:', lock.get('revision'))
for package in lock['package']:
    if package['name'] in {'fastapi', 'starlette', 'anyio'}:
        print(package['name'], package['version'], [dep['name'] for dep in package.get('dependencies', [])])
PY

#### Bob Repairs a Broken Pin

> **Bob:** What if someone writes a version that doesn't exist? I'd rather catch that here than during deployment.

Make an impossible HTTPX pin and demand an expected resolver failure. Then restore the section's `0.27.2` pin and re-lock. Neither experiment installs anything:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.27.2/httpx==999.0.0/' pyproject.toml
if uv lock; then
    echo 'ERROR: impossible pin unexpectedly resolved' >&2
    exit 1
else
    echo 'Expected: resolver rejected the impossible pin.'
fi
sed -i 's/httpx==999.0.0/httpx==0.27.2/' pyproject.toml
uv lock

### Upgrade the Lockfile

> **Bob:** Maintenance day. Which versions can move depends on my constraints, not on wishful thinking.

First try upgrading the exact HTTPX pin: it cannot move past `0.27.2`. Then upgrade FastAPI within its `>=0.115,<0.116` range, retaining other locked versions where possible. Finally ask uv to refresh *all* versions allowed by the declared constraints:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade-package httpx
uv lock --upgrade-package fastapi
uv lock --upgrade
uv sync

### Validate the Lockfile

> **Bob:** Suppose a colleague changes `pyproject.toml` but forgets the lockfile. I want an alarm for that.

Change the FastAPI range to a different compatible family without re-locking. `uv lock --check` should fail without changing the lockfile. Restore the range and watch it pass again. These checks are useful in CI or before committing:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.115,<0.116/fastapi>=0.116,<0.117/' pyproject.toml
lock_before=$(sha256sum uv.lock)
if uv lock --check; then
    echo 'ERROR: check accepted a stale lockfile' >&2
    exit 1
else
    echo 'Expected: check detected drift.'
fi
test "$lock_before" = "$(sha256sum uv.lock)"
sed -i 's/fastapi>=0.116,<0.117/fastapi>=0.115,<0.116/' pyproject.toml
uv lock --check

## Synchronizing

### Default Synchronization

> **Bob:** Lock is truth; `.venv` is reality. Now I make reality match.

`uv sync` creates `.venv` if necessary and installs, changes, or removes packages to match the lockfile. By default it also re-locks stale declarations. Runtime dependencies and `dev` are included by default; `--group dev` selects that group explicitly, and `--all-groups` includes `docs` as well (but not optional extras). The main metadata-only project has no `[build-system]`, so uv installs its dependencies, not the project itself:

In [ ]:
%%bash
set -euxo pipefail
uv sync
uv sync --group dev
uv sync --all-groups
.venv/bin/python -c 'from importlib.metadata import version; print("Ruff:", version("ruff"), "MkDocs:", version("mkdocs"))'

### Frozen Installs

> **Bob:** If the declarations drift, will `--frozen` still use the *old* lockfile? Let's measure it.

Change the FastAPI declaration to the 0.116 family but leave `uv.lock` alone. A frozen sync skips freshness checks and installs the locked 0.115 release anyway. Compare the lockfile checksum and read the installed version *directly* from `.venv` (another uv command could re-lock automatically). This is useful in intermediate Docker layers when the full project isn't yet available; `--no-install-project` skips the project, and `--no-install-workspace` skips workspace projects:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.115,<0.116/fastapi>=0.116,<0.117/' pyproject.toml
lock_before=$(sha256sum uv.lock)
uv sync --frozen
test "$lock_before" = "$(sha256sum uv.lock)"
.venv/bin/python -c 'from importlib.metadata import version; installed = version("fastapi"); print("Installed FastAPI:", installed); assert installed.startswith("0.115.")'
echo 'Frozen sync left the lockfile and installed FastAPI on the old version.'

Bob restores his declaration before the next experiment. In an intermediate build, validate separately and run a final locked sync once all project files are present:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.116,<0.117/fastapi>=0.115,<0.116/' pyproject.toml
uv lock --check

### Locked Installs

> **Bob:** In CI I want zero surprises: install the reviewed lockfile, or fail loudly.

Repeat the *same* FastAPI drift. `uv sync --locked` must reject it rather than edit the lockfile. After that, explicitly re-lock, sync, and check that the environment now contains a 0.116 release. Locked installs suit GitHub Actions and Docker build steps:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.115,<0.116/fastapi>=0.116,<0.117/' pyproject.toml
lock_before=$(sha256sum uv.lock)
if uv sync --locked; then
    echo 'ERROR: locked sync accepted a stale lockfile' >&2
    exit 1
else
    echo 'Expected: locked sync rejected the stale lockfile.'
fi
test "$lock_before" = "$(sha256sum uv.lock)"
uv lock
uv sync --locked
.venv/bin/python -c 'from importlib.metadata import version; installed = version("fastapi"); print("Installed FastAPI:", installed); assert installed.startswith("0.116.")'

Return to the section's FastAPI range and re-lock explicitly so the rest of Bob's lab starts with matching files:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.116,<0.117/fastapi>=0.115,<0.116/' pyproject.toml
uv lock
uv sync --locked

#### Bob Tests a Non-Editable Install

> **Bob:** My dependencies are locked. What about *my own code*? If I edit source after installation, does production change under my feet?

A project needs a `[build-system]` for uv to install it. Make a separate, dependency-free `installed-demo/` with its own lockfile and virtual environment. Normally uv installs a project editably; `--no-editable` copies built package files into `site-packages`. Test the installed version with isolated Python (`-I`) to avoid accidentally importing the source checkout:

In [ ]:
%%bash
set -euxo pipefail
mkdir -p installed-demo/src/installed_demo
cat > installed-demo/pyproject.toml <<'EOF'
[project]
name = "installed-demo"
version = "0.1.0"
requires-python = ">=3.11"

[build-system]
requires = ["setuptools==80.9.0"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]
EOF
cat > installed-demo/src/installed_demo/__init__.py <<'EOF'
MESSAGE = "Original source"
EOF
uv lock --project installed-demo
uv sync --project installed-demo --locked --no-editable
installed-demo/.venv/bin/python -I -c 'import installed_demo; print(installed_demo.MESSAGE, installed_demo.__file__); assert installed_demo.MESSAGE == "Original source"'

Edit the source file. The next isolated import still reads the installed copy, not the edited source. A regular sync may reuse the installation, so ask for an explicit reinstall to apply the edit:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/Original source/Edited source/' installed-demo/src/installed_demo/__init__.py
cat installed-demo/src/installed_demo/__init__.py
installed-demo/.venv/bin/python -I -c 'import installed_demo; print("Before reinstall:", installed_demo.MESSAGE); assert installed_demo.MESSAGE == "Original source"'
uv sync --project installed-demo --locked --no-editable --reinstall-package installed-demo
installed-demo/.venv/bin/python -I -c 'import installed_demo; print("After reinstall:", installed_demo.MESSAGE); assert installed_demo.MESSAGE == "Edited source"'

### Resolution Strategies

> **Bob:** If my library says it supports older versions, I'd better test those too — the newest install only proves the newest combination.

uv normally selects the latest compatible versions. `lowest` chooses the lowest compatible direct *and* transitive versions; `lowest-direct` lowers only direct dependencies. For this comparison Bob uses a tiny separate project with a range, so the main lockfile and the earlier Git source are unaffected. He pins its Python requirement to the container's minor version to keep the experiment local:

In [ ]:
%%bash
set -euxo pipefail
mkdir -p resolution-demo
python3 - <<'PY'
import sys
from pathlib import Path
Path('resolution-demo/pyproject.toml').write_text('[project]\nname = "resolution-demo"\nversion = "0.1.0"\nrequires-python = "==%s.%s.*"\ndependencies = ["fastapi>=0.115,<0.116"]\n' % sys.version_info[:2])
PY
uv sync --project resolution-demo --resolution lowest --no-install-project
resolution-demo/.venv/bin/python -c 'from importlib.metadata import version; print("Lowest FastAPI:", version("fastapi"), "Starlette:", version("starlette"))'
uv sync --project resolution-demo --resolution lowest-direct --no-install-project
resolution-demo/.venv/bin/python -c 'from importlib.metadata import version; print("Lowest-direct FastAPI:", version("fastapi"), "Starlette:", version("starlette"))'

#### Inspect the Graph

> **Bob:** Which dependency brought Starlette in? Time for a family tree.

`uv tree` displays direct and transitive dependencies from the main project. Like other project commands, it may refresh an outdated lockfile; Bob has already restored his declarations:

In [ ]:
%%bash
set -euxo pipefail
uv tree
uv lock --check

## Export Dependencies

### Share a Requirements File

> **Bob:** Some colleagues still need a `requirements.txt`. I can share one without abandoning the lockfile.

Export from the main project's lockfile. Regenerate this file whenever dependencies change; `pyproject.toml` and `uv.lock` remain the source of truth. For working directly with legacy requirements files, see [Standalone Environments and Tools](../../docs/chapter-03/section-05.md#work-with-legacy-projects):

In [ ]:
%%bash
set -euxo pipefail
uv export --format requirements.txt -o requirements.txt --quiet
sed -n '1,18p' requirements.txt

## Epilogue — Bob's Runbook Notes

> **Bob:** Alice gets a coffee for that sticky note. My runbook now records intent, truth, reality — and what to do when they disagree.

| File | Changed by | Read by |
| --- | --- | --- |
| `pyproject.toml` | You, `uv add`, `uv remove` | `uv lock` |
| `uv.lock` | `uv lock`, `uv add`, `uv remove`, and automatic re-locking by `uv sync` | `uv sync`, `uv lock --check` |
| `.venv` | `uv sync` (also `uv add` and `uv remove`) | Your application |

1. **Declare intent.** Keep runtime requirements, groups, and source locations in `pyproject.toml`.
2. **Review truth.** Commit `uv.lock`; use `uv lock --check` to catch drift and `uv lock --upgrade-package` or `--upgrade` to refresh it deliberately.
3. **Control reality.** `uv sync` aligns the environment, `--frozen` skips freshness checks, and `--locked` rejects stale locks. For deployed code, `--no-editable` installs built files until explicitly reinstalled.